# 数据准备与基线训练

前两章我们理解了 SFT 概念、Wordle 任务和 TorchTitan 框架。环境也已配置完毕。

本章将走完"训练配置构建 → SFT 训练 → 推理评测"的完整闭环，获得你的第一个 Wordle SFT 模型。03.02 从数据集出发，逐层深入训练配置的每个参数背后的概念；03.03 动手跑训练；03.04 验证效果。

---

## 教程进度回顾

| 章节 | 内容 | 状态 |
|------|------|------|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| **第 3 章** | **训练配置 + 基线训练 + 推理评测** | ← 当前 |
| 第 4 章 | 融合算子 + Profiling 优化 | 待开始 |

---

## 本章三步闭环

![Chapter 3 Pipeline](./images/chapter3-pipeline.png)

文字版流程：先检查 Wordle 样本如何变成 masked labels，再在两个 die 对应的两个 rank 上运行 20 个 optimizer steps，最后分别加载基座模型与 SFT checkpoint，在同一评测集合上保存逐局结果。


---

## 前置条件

- 第 2 章环境已配置（`torchtitan-npu` 可正常 `import`）。
- Ascend NPU 可用（`npu-smi info` 正常）。
- Qwen3-1.7B 基座模型权重已下载到 `./assets/hf/Qwen3-1.7B/`。

---

## 本章产物与验收

本章不预先规定 loss、reward 或耗时数值。完成后应得到以下可检查产物：

| 产物 | 验收条件 |
|------|----------|
| 训练初始化日志 | `NGPU=2` 启动两个 rank；最终配置显示 DP degree=2、梯度累计 16 次 |
| 20-step 训练日志 | loss 与裁剪前 `grad_norm` 均为有限值；进程以 0 退出 |
| SFT checkpoint | 目录包含 HF 权重及所需 config/tokenizer assets，且能用 `from_pretrained()` 重新加载 |
| base/SFT 评测结果 | 相同评测 revision、任务集合、shuffle seed 和生成参数；逐局结果已保存且没有请求错误 |

评测时单独报告格式、正确答案、部分奖励和实际 guess 数。合法协议为 `<think>...</think><guess>[word]</guess>`；`correct_answer` 还会精确检查方括号。任何“格式改善”或“策略改善”结论都必须从本次保存的两组结果计算。